# Notebook 4 — Which Features Best Predict Sale Price?

In this notebook we go one level deeper and answer a simple but important question:  
**Which house features have the strongest connection to the sale price?**

We will:
1. Load the cleaned data
2. Measure how strongly each feature is connected to price (correlation)
3. Visualise the top predictors
4. Pick the single best feature and show why it works so well

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned and engineered data saved by notebook 02
# This gives us all the original columns PLUS the new features we created
df_clean = pd.read_pickle("data/engineered/ames_engineered.pkl")

print(f"Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")
df_clean.head()

---
## Step 1 — Measure Correlation with Sale Price

**Correlation** is a number between -1 and 1 that tells us how closely two things move together:
- **Close to 1** → when one goes up, the other goes up too (strong positive link)
- **Close to -1** → when one goes up, the other goes down (strong negative link)
- **Close to 0** → no real connection

We calculate the correlation of every numeric feature with `SalePrice` and rank them.

In [ ]:
# Keep only numeric columns
df_numeric = df_clean.select_dtypes(include=['number'])

# Calculate correlation of every feature with SalePrice
# We drop SalePrice itself and SalePrice_log — the log version is just a transformation
# of the target, so it would always rank #1 and give a misleading result
correlations = df_numeric.corr()['SalePrice'].drop(['SalePrice', 'SalePrice_log'], errors='ignore')

# Sort by absolute value — strongest connection first
correlations_sorted = correlations.abs().sort_values(ascending=False)

# Show top 10
print('Top 10 features most connected to Sale Price:')
print(correlations_sorted.head(10))

---
## Step 2 — Bar Chart of the Top 10 Predictors

A bar chart makes it easy to compare the strength of each feature at a glance.
The longer the bar, the stronger the connection to price.

In [ ]:
top10 = correlations_sorted.head(10)

# [::-1] reverses the list so the strongest feature appears at the top of the chart
top10_reversed = top10[::-1]

plt.figure(figsize=(10, 6))
plt.barh(top10_reversed.index, top10_reversed.values, color='steelblue', edgecolor='black')
plt.xlabel('Correlation with Sale Price (absolute value)')
plt.title('Top 10 Features That Best Predict Sale Price')
plt.tight_layout()
plt.show()

# Get the name of the best feature (first item in the ranked list)
best_feature = top10.index[0]

# Look up its actual correlation value (not the absolute version) to keep the +/- sign
best_score = correlations[best_feature]

# :.2f means "show only 2 decimal places" — e.g. 0.79 instead of 0.790123...
print(f'Best single predictor: {best_feature} (r = {best_score:.2f})')

---
## Step 3 — Focus on the Best Predictor

**Overall Quality** is the strongest predictor. It is a rating from 1 to 10 given to each house.
Let us see the average sale price at each quality level — does price go up steadily as quality increases?

In [ ]:
# Average sale price at each quality level
quality_price = df_clean.groupby('Overall Qual')['SalePrice'].mean()

plt.figure(figsize=(10, 5))
plt.plot(quality_price.index, quality_price.values, marker='o', color='darkorange', linewidth=2, markersize=8)
plt.xlabel('Overall Quality (1 = Very Poor, 10 = Excellent)')
plt.ylabel('Average Sale Price ($)')
plt.title('How Does Quality Rating Affect Sale Price?')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Print the difference between lowest and highest quality
low  = quality_price.iloc[0]
high = quality_price.iloc[-1]
print(f'Average price at quality 1:  ${low:,.0f}')
print(f'Average price at quality 10: ${high:,.0f}')
print(f'Difference: ${high - low:,.0f}')

---
## Step 4 — Weak Predictors

Not every feature is useful. Let us also look at the **bottom 5** — features with almost no connection to price.
Knowing what does NOT matter is just as useful as knowing what does.

In [ ]:
bottom5 = correlations_sorted.tail(5)

print('5 features with the WEAKEST connection to Sale Price:')
print(bottom5)

print('\nThese features tell us almost nothing about the price.')
print('A model would be better off ignoring them.')

---
## Conclusion

From this deeper exploration we found that:

- **Overall Quality** is the single best predictor of sale price — a house rated 10 sells for dramatically more than one rated 1.
- **Living Area**, **Garage Size**, and **Basement Area** are also strongly connected to price — bigger generally means more expensive.
- Some features like `MS SubClass` or `Mo Sold` have almost no connection to price and could be safely removed from a prediction model.

If we were building a simple price predictor, we would start with just the top 5 features and already get a very strong result.